# ENTERPRISE KNOWLEDGE MINING SYSTEM

#### DOCUMENT INGESTION

In [9]:
import xml.etree.ElementTree as ET
import time
import pymupdf4llm
import os
import re

BASE_URL = "https://export.arxiv.org/api/query"

# arXiv uses Atom namespace
NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom"    # extension elements that are not mapped to standard Atom specification
}

params = {
    "search_query": "all:computer",
    "start": 0,
    "max_results": 2
}

In [10]:
import urllib.request
import urllib.error

# ==== Making Request ====
def make_request(url, delay=5, retries=3):
    for attempt in range(retries):
        try:
            time.sleep(delay)
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "arxiv-data-project/1.0 (research script)"}
            )
            with urllib.request.urlopen(request) as response:
                return response.read()

        except urllib.error.HTTPError as e:
            if e.code == 429:
                wait = delay * (2 ** attempt)
                time.sleep(wait)
            else:
                raise

    raise Exception("Max retries exceeded")

In [11]:
# ==== Parsing Helper Functions ====
def get_text(entry, tag):
    return entry.findtext(tag, default="", namespaces=NS).strip()


def extract_authors(entry):
    return [
        author.find("atom:name", NS).text.strip()
        for author in entry.findall("atom:author", NS)
    ]


def extract_categories(entry):
    return [
        cat.attrib.get("term")
        for cat in entry.findall("atom:category", NS)
        if cat.attrib.get("term")
    ]


def extract_pdf_link(entry):
    return next(
        (
            link.attrib.get("href")
            for link in entry.findall("atom:link", NS)
            if link.attrib.get("title") == "pdf"
            and link.attrib.get("type") == "application/pdf"
            and link.attrib.get("rel") == "related"
        ),
        None
    )


def extract_primary_category(entry):
    primary = entry.find("arxiv:primary_category", NS)
    return primary.attrib.get("term") if primary is not None else None


def extract_paper(entry):
    return {
        "arxiv_id": get_text(entry, "atom:id").split("/")[-1],
        "title": get_text(entry, "atom:title"),
        "summary": get_text(entry, "atom:summary"),
        "published": get_text(entry, "atom:published"),
        "authors": extract_authors(entry),
        "primary_category": extract_primary_category(entry),
        "categories": extract_categories(entry),
        "pdf_link": extract_pdf_link(entry),
    }

In [12]:
def parse_response(response):
    root = ET.fromstring(response)
    return [extract_paper(entry) for entry in root.findall("atom:entry", NS)]

#### EXTRACTING PDF AND CONVERTING TO MARKDOWN

In [13]:
pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok = True)


def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)


def download_pdf(url, save_path, overwrite=False):
    if os.path.exists(save_path) and not overwrite:
        return save_path

    urllib.request.urlretrieve(url, save_path)
    return save_path


def process_paper(paper):
    if not paper.get("pdf_link"):
        print(f"Skipping {paper['title']} (no PDF)")
        return None
    
    safe_title = sanitize_filename(paper["title"])
    local_pdf = f"{safe_title}.pdf"
    path = os.path.join(pdf_dir, local_pdf)

    pdf_file = download_pdf(paper["pdf_link"], path)
    pages = pymupdf4llm.to_markdown(pdf_file, page_chunks=True)

    return {
        "arxiv_id": paper["arxiv_id"],
        "title": paper["title"],
        "pages": pages
    }

#### DATA CLEANING FOR CHUNKING

In [14]:
# cleaning functions for headings
def clean_heading(text: str) -> str:
    text = text.strip()
    text = re.sub(r'[*_`#]+', '', text)   # remove markdown markers
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [15]:
# splitting pages into section blocks based on headings and common section titles
def extract_section_blocks(page_text: str):
    pattern = re.compile(
        r'^(##.*|_?\*{0,2}Abstract\*{0,2}_?.*|_?\*{0,2}Keywords:.*\*{0,2}_?)$',
        re.IGNORECASE | re.MULTILINE
    )

    matches = list(pattern.finditer(page_text))
    section_blocks = []

    for i, match in enumerate(matches):
        section = re.sub(r'[*_`#]+', '', match.group()).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(page_text)
        text = page_text[start:end].strip()

        if text:
            section_blocks.append({
                "section": section,
                "text": text
            })

    return section_blocks

In [16]:
def clean_text(text: str) -> str:
    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove markdown heading markers and separator-only lines
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)

    # Remove markdown emphasis symbols anywhere in text
    text = re.sub(r"[_*`~]+", " ", text)

    # Remove hash symbols that can confuse NER
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")

    # Fix hyphenated line breaks: learn-\ning -> learning
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)

    # Remove obvious page-only lines
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

    # Normalize long dashes
    text = re.sub(r"[—–]+", " - ", text)

    # Remove leading punctuation/whitespace 
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    # Collapse repeated whitespace/newlines
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")

    # Final whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [17]:
def clean_page_text(text: str, page_number=None) -> str:
    text = clean_text(text)

    if not text:
        return ""

    # First-page specific cleanup:
    # if Abstract exists, keep text starting from Abstract
    if page_number == 1:
        match = re.search(r"\babstract\b\s*[:-]?\s*", text, flags=re.IGNORECASE)
        if match:
            text = text[match.start():]

    return re.sub(r"\s+", " ", text).strip()

In [18]:
def clean_processed_papers(processed_papers, min_length):
    cleaned_papers = []
    
    for paper in processed_papers:
        clean_pages = []

        for page in paper["pages"]:
            cleaned = clean_page_text(
                page["text"],
                page_number=page["metadata"].get("page_number")
            )

            if cleaned and len(cleaned) >= min_length:  # filter out very short/empty pages
                clean_pages.append({
                    "text": cleaned,
                    "metadata": page["metadata"]
                })

        cleaned_papers.append({
            "arxiv_id": paper["arxiv_id"],
            "title": paper["title"],
            "pages": clean_pages
        })
        
    return cleaned_papers

In [19]:
import urllib.parse
from debug_utils import preview_cleaned_paper

def process_pdf_to_chunks(params, base_url=BASE_URL, min_length=50):
    query_url = base_url + "?" + urllib.parse.urlencode(params)

    response = make_request(query_url)

    papers = parse_response(response)

    processed_papers = []
    for paper in papers:
        result = process_paper(paper)
        if result:
            processed_papers.append(result)

    cleaned_papers = clean_processed_papers(processed_papers, min_length)

    return cleaned_papers   

cleaned_papers = process_pdf_to_chunks(params)
preview_cleaned_paper(cleaned_papers)

OCR disabled because Tesseract language data not found.
OCR disabled because Tesseract language data not found.

==== Vision Based Game Development Using Human Computer Interaction ====

--- Page 0 Preview ---
Metadata: {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using Human Computer Interaction.pdf', 'page_count': 7, 'page_number': 1}
Text Preview:
 Abstract - A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between ma

In [24]:
# extracting section blocks from cleaned pages
all_blocks = []

for page in pages:
    page_blocks = extract_section_blocks(page["text"])
    for block in page_blocks:
        block["text"] = clean_text(block["text"])
        block["page_number"] = page["metadata"].get("page_number")
        all_blocks.append(block)

In [25]:
for block in all_blocks:
    print(f"\nPAGE: {block['page_number']}")
    print(f"SECTION: {block['section']}")
    print(f"TEXT SAMPLE: {block['text'][:500]}")

print(f"\nTotal sections extracted: {len(all_blocks)}")


PAGE: 1
SECTION: Vision Based Game Development Using Human Computer Interaction
TEXT SAMPLE: Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India

PAGE: 1
SECTION: I. INTRODUCTION
TEXT SAMPLE: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen. The tip of the user's nose can be tracked and captured with a webcam and monitor its movements in order to translate it 

PAGE: 2
SECTION: II. RELATED WORK
TEXT SAMPLE: With the growth of attention about computer vision, the interest in HCI has increased proportionally. Different human features and monitoring devices were used to achieve HCI, but during our resear

In [26]:
#chunking the cleaned text for spacy using langchain text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,  #may need to reduce when scaling up to avoid too much redundancy
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""]
)

paper = papers[0] if papers else {}

paper_id = paper.get("arxiv_id", "paper_0")   
title = paper.get("title", "untitled")
authors = ", ".join(paper.get("authors", [])) if paper.get("authors") else ""
published = paper.get("published", "")
abstract = paper.get("summary", "")
primary_category = paper.get("primary_category", "")
categories = paper.get("categories", [])
categories_str = ", ".join(categories) if isinstance(categories, list) else str(categories)

chunks = []
global_chunk_index = 0

normalized_title = title.strip().lower()

#excluding sections that may not be useful for NER
EXCLUDED_SECTIONS = {
    "References",
    "Acknowledgment",
    "Acknowledgements"
}

for page in pages:
    page_number = page["metadata"].get("page_number", "")
    page_count = page["metadata"].get("page_count", "")
    file_format = page["metadata"].get("format", "")
    creation_date = page["metadata"].get("creationDate", "")

    page_blocks = extract_section_blocks(page["text"])

    #filtering title since its already in the metadata
    filtered_blocks = []
    for block in page_blocks:
        section_name = block.get("section", "").strip()

        if section_name.lower() == normalized_title:
            continue

        if section_name.title() in EXCLUDED_SECTIONS:
            continue

        filtered_blocks.append(block)

    for block_idx, block in enumerate(filtered_blocks):
        section = block.get("section", "Unknown")
        block_text = clean_text(block.get("text", ""))

        if not block_text.strip():
            continue

        splits = text_splitter.split_text(block_text)

        for page_chunk_index, s in enumerate(splits):
            s = re.sub(r"^[\s\.,;:-]+", "", s)

            if s.strip():
                chunks.append({
                    "text": s,
                    "metadata": {
                        "paper_id": paper_id,
                        "title": title,
                        "section": section,
                        "primary_category": primary_category,
                        "categories": categories_str,
                        "authors": authors,
                        "published": published,
                        "creation_date": creation_date,
                        "page_number": page_number,
                        "page_count": page_count,
                        "format": file_format,
                        "page_block_index": block_idx,
                        "page_chunk_index": page_chunk_index,
                        "chunk_index": global_chunk_index
                    }
                })
                global_chunk_index += 1

print("Total chunks:", len(chunks))
print(chunks[0] if chunks else "No chunks created")

Total chunks: 45
{'text': "One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen", 'metadata': {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0}}


In [27]:
# separating metadata from text for NER to avoid noise
def text_for_ner(chunk):
    text = (chunk.get("text") or "")
    md = chunk.get("metadata", {})

    # Clean special chars that confuse NER
    text = re.sub(r"[*_`~^—–]+", " ", text)

    # Remove leading punctuation/whitespace
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    if md.get("page_number") == 1:
        title = (md.get("title") or "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)

        authors = md.get("authors", "")
        if isinstance(authors, str):
            author_list = [a.strip() for a in authors.split(",") if a.strip()]
        elif isinstance(authors, list):
            author_list = [str(a).strip() for a in authors if str(a).strip()]
        else:
            author_list = []

        for author in author_list:
            text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)

        # Keep only text after Abstract if present
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]

    return re.sub(r"\s+", " ", text).strip()

#### Entity Extraction

In [28]:
#using spacy model for NER

import spacy

nlp = spacy.load("en_core_web_md") # using at the moment since it has better NER performance than the small mode
# nlp = spacy.load("en_core_web_sm") # would eventually switch when increase research papers for faster performance

In [29]:
def clean_entities(raw_entities):
    cleaned = []
    seen = set()

    for ent_text, ent_label in raw_entities:
        ent_text = ent_text.strip()

        # skip empty entities
        if not ent_text:
            continue

        # remove very short fragments like "I."
        if len(ent_text) < 3:
            continue

        # remove pure numbers
        if re.fullmatch(r"\d+(\.\d+)?", ent_text):
            continue

        # skip ordinal/cardinal noise
        if ent_label in {"CARDINAL", "ORDINAL"}:
            continue

        # remove mostly punctuation fragments
        if re.fullmatch(r"[^\w]+", ent_text):
            continue

        key = (ent_text.lower(), ent_label)
        if key not in seen:
            seen.add(key)
            cleaned.append((ent_text, ent_label))

    return cleaned

In [30]:
# updating metadata with NER results
ner_texts = [text_for_ner(c) for c in chunks]
docs = nlp.pipe(ner_texts, batch_size=32)

for i, doc in enumerate(docs):
    raw_entities = [(ent.text, ent.label_) for ent in doc.ents]
    cleaned_entities = clean_entities(raw_entities)

    chunks[i]["metadata"]["entities_raw"] = raw_entities
    chunks[i]["metadata"]["entities_clean"] = cleaned_entities
    chunks[i]["metadata"]["entities"] = ", ".join([ent_text for ent_text, _ in cleaned_entities])
    chunks[i]["metadata"]["entity_labels"] = ", ".join([ent_label for _, ent_label in cleaned_entities])
    chunks[i]["metadata"]["ner_text"] = ner_texts[i]

    if i < 15:
        print("\nChunk", i, "section:", chunks[i]["metadata"].get("section"))
        print("Chunk", i, "primary_category:", chunks[i]["metadata"].get("primary_category"))
        print("Chunk", i, "categories:", chunks[i]["metadata"].get("categories"))
        print("Chunk", i, "cleaned entities:", cleaned_entities[:20])
        print("Chunk", i, "NER text preview:", ner_texts[i][:180])


Chunk 0 section: I. INTRODUCTION
Chunk 0 primary_category: cs.HC
Chunk 0 categories: cs.HC, cs.CV, cs.MM
Chunk 0 cleaned entities: [('HCI', 'ORG')]
Chunk 0 NER text preview: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI a

Chunk 1 section: I. INTRODUCTION
Chunk 1 primary_category: cs.HC
Chunk 1 categories: cs.HC, cs.CV, cs.MM
Chunk 1 cleaned entities: []
Chunk 1 NER text preview: The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen. The tip of the user's nose can be tra

Chunk 2 section: I. INTRODUCTION
Chunk 2 primary_category: cs.HC
Chunk 2 categories: cs.HC, cs.CV, cs.MM
Chunk 2 cleaned entities: []
Chunk 2 NER text preview: In our system, the nose tip is the pointing device, because of the location and shape of the nose, as it is located in the middle of the face

In [31]:
# printing the metadata of the first chunk to verify NER results are stored
print(chunks[0]["metadata"])

{'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0, 'entities_raw': [('One', 'CARDINAL'), ('HCI', 'ORG'), ('HCI', 'ORG')], 'entities_clean': [('HCI', 'ORG')], 'entities': 'HCI', 'entity_labels': 'ORG', 'ner_text': "One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen

In [32]:
final_chunks = []

for i, chunk in enumerate(chunks):
    final_chunks.append({
        "text": chunk["text"],
        "metadata": {
            "paper_id": chunk["metadata"].get("paper_id", "paper_0"),
            "title": chunk["metadata"].get("title", ""),
            "section": chunk["metadata"].get("section", ""),
            "primary_category": chunk["metadata"].get("primary_category", ""),
            "categories": chunk["metadata"].get("categories", ""),
            "authors": chunk["metadata"].get("authors", ""),
            "published": chunk["metadata"].get("published", ""),
            "creation_date": chunk["metadata"].get("creation_date", ""),
            "page_number": chunk["metadata"].get("page_number", ""),
            "page_count": chunk["metadata"].get("page_count", ""),
            "format": chunk["metadata"].get("format", ""),
            "page_block_index": chunk["metadata"].get("page_block_index", ""),
            "page_chunk_index": chunk["metadata"].get("page_chunk_index", ""),
            "chunk_index": chunk["metadata"].get("chunk_index", i),
            "entities": chunk["metadata"].get("entities", ""),
            "entity_labels": chunk["metadata"].get("entity_labels", "")
        }
    })

    if i < 5:
        print("\nFinal Chunk", i, "metadata:", final_chunks[-1]["metadata"])
        print("Final Chunk", i, "text preview:", final_chunks[-1]["text"][:180])


Final Chunk 0 metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0, 'entities': 'HCI', 'entity_labels': 'ORG'}
Final Chunk 0 text preview: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI a

Final Chunk 1 metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, 

#### EMBEDDING WITH OPENAI AND STORING IN CHROMADB

In [33]:
from openai import OpenAI
import os
import dotenv
import chromadb

dotenv.load_dotenv()  # Load environment variables from .env file

client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
embedding_model = "text-embedding-3-small" # will change to text-embedding-3-large when increase research papers for better performance

# using persistent chroma db to store the embeddings and metadata for later retrieval and analysis
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection_name = "research_papers"

try:
    chroma_client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print(f"No existing collection named: {collection_name}")

collection = chroma_client.get_or_create_collection(
    name=collection_name,
    configuration={"hnsw": {"space": "cosine"}}
)
print("Chroma collection ready.")

No existing collection named: research_papers
Chroma collection ready.


In [34]:
#preparing data for insertion into chroma db
documents = [chunk["text"] for chunk in final_chunks]
metadatas = [chunk["metadata"] for chunk in final_chunks]
ids = [
    f"{chunk['metadata']['paper_id']}_chunk_{chunk['metadata']['chunk_index']}"
    for chunk in final_chunks
]

print("Documents:", len(documents))
print("Metadatas:", len(metadatas))
print("IDs:", len(ids))
print("Sample ID:", ids[0] if ids else "N/A")
print("Sample metadata:", metadatas[0] if metadatas else "N/A")

Documents: 45
Metadatas: 45
IDs: 45
Sample ID: 1002.2191v1_chunk_0
Sample metadata: {'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'page_block_index': 0, 'page_chunk_index': 0, 'chunk_index': 0, 'entities': 'HCI', 'entity_labels': 'ORG'}


In [35]:
#batching embedding
def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

In [36]:
batch_size = 20
all_embeddings = []

for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i + batch_size]
    batch_embeddings = get_embeddings(batch_docs, model=embedding_model)
    all_embeddings.extend(batch_embeddings)
    print(f"Embedded chunks {i} to {i + len(batch_docs) - 1}")

print("Total embeddings created:", len(all_embeddings))
print("Embedding dimension:", len(all_embeddings[0]) if all_embeddings else 0)

Embedded chunks 0 to 19
Embedded chunks 20 to 39
Embedded chunks 40 to 44
Total embeddings created: 45
Embedding dimension: 1536


In [37]:
#storing embeddings and metadata in chroma db
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=all_embeddings
)

print(f"Stored {len(documents)} chunks in Chroma collection '{collection_name}'.")

Stored 45 chunks in Chroma collection 'research_papers'.


### Testing Retrieval

In [38]:
def retrieve_chunks(query, n_results=3):
    query_embedding = get_embeddings([query], model=embedding_model)[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results

In [39]:
test_queries = [
    "What is the main purpose of the proposed HCI system?",
    "What is Hough transform and how is it used in this paper?",
    "How does blink detection work in the eye ROI?"
]

for query in test_queries:
    print("\n" + "="*50)
    print(f"\nSearch results for query: '{query}'")
    results = retrieve_chunks(query, n_results=3)

    for i in range(len(results["documents"][0])):
        print(f"\nResult {i+1}")
        print("ID:", results["ids"][0][i])
        print("Metadata:", results["metadatas"][0][i])
        print("Text:", results["documents"][0][i][:400])
    print("\n" + "="*50)



Search results for query: 'What is the main purpose of the proposed HCI system?'

Result 1
ID: 1002.2191v1_chunk_0
Metadata: {'entities': 'HCI', 'page_chunk_index': 0, 'categories': 'cs.HC, cs.CV, cs.MM', 'primary_category': 'cs.HC', 'page_block_index': 0, 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'chunk_index': 0, 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'format': 'PDF 1.4', 'page_count': 7, 'section': 'I. INTRODUCTION', 'page_number': 1, 'entity_labels': 'ORG', 'paper_id': '1002.2191v1'}
Text: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer 

### SEMANTIC SEARCH ENGINE


In [40]:
#entity normalization function to create a consistent set of entities for analysis and retrieval
def normalize_entity_set(entities: str):
    return {
        e.strip().lower()
        for e in entities.split(",")
        if e.strip()
    }

In [41]:
# entity overlap function to compute the number of shared entities between a chunk and a query for relevance scoring and ranking
def compute_entity_overlap(entities: str, query: str):
    entity_set = normalize_entity_set(entities)
    query_terms = {term.lower().strip() for term in query.split() if term.strip()}
    return len(entity_set.intersection(query_terms))

In [57]:
# hybrid scoring to combine cosine similarity with entity overlap for improved relevance ranking of search results
def compute_hybrid_score(cosine_similarity, entity_overlap, use_hybrid=False, entity_bonus=0.5):
    if cosine_similarity is None:
        return None
    if not use_hybrid:
        return cosine_similarity
    return cosine_similarity + (entity_overlap * entity_bonus)

In [58]:
def semantic_search(query, n_results=5, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    
    results = retrieve_chunks(query, n_results=n_results)
    formatted_results = []

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []

    # this filter is for experimentation to improve llm response accuracy (the idea is if the user specify a category, filter the chunks to only that category to reduce noise for the llm)
    filter_category_norm = filter_category.lower().strip() if filter_category else None

    for i, doc_text in enumerate(documents):
        metadata = metadatas[i] if i < len(metadatas) else {}
        chunk_id = ids[i] if i < len(ids) else f"chunk_{i}"

        cosine_distance = distances[i] if i < len(distances) else None
        cosine_similarity = 1 - cosine_distance if cosine_distance is not None else None

        if cosine_similarity is not None and cosine_similarity < min_similarity:
            continue

        primary_category = metadata.get("primary_category", "")
        if filter_category_norm and primary_category.lower().strip() != filter_category_norm:
            continue

        entities = metadata.get("entities", "")
        print(f"\nEntities for chunk {chunk_id}: {entities}")

        entity_overlap = compute_entity_overlap(entities, query)
        hybrid_score = compute_hybrid_score(
            cosine_similarity,
            entity_overlap,
            use_hybrid=use_hybrid,
            entity_bonus=entity_bonus
        )

        formatted_results.append({
            "rank": i + 1,
            "chunk_id": chunk_id,
            "cosine_distance": cosine_distance,
            "cosine_similarity": cosine_similarity,
            "hybrid_score": hybrid_score,
            "entity_overlap": entity_overlap,
            "text": doc_text,
            "paper_id": metadata.get("paper_id", ""),
            "title": metadata.get("title", ""),
            "section": metadata.get("section", ""),
            "primary_category": primary_category,
            "categories": metadata.get("categories", ""),
            "authors": metadata.get("authors", ""),
            "published": metadata.get("published", ""),
            "page_number": metadata.get("page_number", ""),
            "page_block_index": metadata.get("page_block_index", ""),
            "page_chunk_index": metadata.get("page_chunk_index", ""),
            "chunk_index": metadata.get("chunk_index", ""),
            "entities": entities,
            "entity_labels": metadata.get("entity_labels", "")
        })

    if use_hybrid:
        formatted_results = sorted(
            formatted_results,
            key=lambda x: x["hybrid_score"] if x["hybrid_score"] is not None else -1,
            reverse=True
        )
        for idx, result in enumerate(formatted_results, start=1):
            result["rank"] = idx

    return formatted_results


In [44]:
def display_search_results(results, preview_chars=300):
    if not results:
        print("No results found.")
        return

    for r in results:
        print("=" * 80)
        print(f"Rank: {r['rank']}")
        print(f"Paper ID: {r['paper_id']}")
        print(f"Title: {r['title']}")
        print(f"Section: {r['section']}")
        print(f"Primary Category: {r['primary_category']}")
        print(f"Categories: {r['categories']}")
        print(f"Authors: {r['authors']}")
        print(f"Published: {r['published']}")
        print(
            f"Page: {r['page_number']} | "
            f"Block: {r['page_block_index']} | "
            f"Page Chunk: {r['page_chunk_index']} | "
            f"Global Chunk: {r['chunk_index']}"
        )
        # print(f"Cosine Distance: {r['cosine_distance']}")
        print(f"Cosine Similarity (approx): {r['cosine_similarity']}")
        print(f"Entities: {r['entities']}")
        print(f"Entity Labels: {r['entity_labels']}")
        print(f"Preview: {r['text'][:preview_chars]}")
        print()

In [45]:
def semantic_search_engine(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    return {
        "query": query,
        "total_results": len(results),
        "results": results
    }

In [46]:
search_output = semantic_search_engine("What is the main purpose of the proposed HCI system?", n_results=3)

print("Query:", search_output["query"])
print("Total results:", search_output["total_results"])
display_search_results(search_output["results"])


Entities for chunk 1002.2191v1_chunk_0: HCI

Entities for chunk 1002.2191v1_chunk_7: HCI

Entities for chunk 1002.2191v1_chunk_12: HCI, EOG
Query: What is the main purpose of the proposed HCI system?
Total results: 3
Rank: 1
Paper ID: 1002.2191v1
Title: Vision Based Game Development Using Human Computer Interaction
Section: I. INTRODUCTION
Primary Category: cs.HC
Categories: cs.HC, cs.CV, cs.MM
Authors: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
Published: 2010-02-10T19:46:07Z
Page: 1 | Block: 0 | Page Chunk: 0 | Global Chunk: 0
Cosine Similarity (approx): 0.6012195944786072
Entities: HCI
Entity Labels: ORG
Preview: One of the promising fields in artificial intelligence is HCI. Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video ca

Rank: 2
Paper ID: 1002.2191v1
Title: Vision Based Game Development Usin

### RAG PIPELINE IMPLEMENTATION

In [47]:
RAG_SYSTEM_PROMPT = """
You are an academic research assistant for an enterprise knowledge mining system built on arXiv papers.

Instructions:
- Answer questions only from the retrieved context.
- Treat the retrieved text as excerpts from research papers.
- Prefer precise academic wording.
- Summarize contributions, methods, datasets, results, or limitations only if they are supported by the context.
- If the answer is incomplete or missing from the context, say that explicitly.
- Do not include source tags or citation brackets such as [Source 1] in the answer text. 
- Write only the answer and ensure it is formatted concisely and clearly.
"""
# not including source tags so the answer is cleaner, the sources will be provided separately in the metadata for each retrieved chunk

In [48]:
# function to generate answer from retrieved context using the RAG system prompt
def generate_answer(query, context, max_tokens=200, temperature=0.2):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use the retrieved research-paper context below to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n\n"
                    "Answer:"
                )
            }
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

In [59]:
# function to perform RAG query and return answer with sources
def rag_query(query, n_results=3, min_similarity=0.25, filter_category=None, use_hybrid=False, entity_bonus=0.5):
    
    search_results = semantic_search(query=query, n_results=n_results, min_similarity=min_similarity, filter_category=filter_category, use_hybrid=use_hybrid, entity_bonus=entity_bonus)

    if not search_results:
        return {
            "query": query,
            "answer": "I cannot answer this question based on the provided context. No relevant context was found.",
            "sources": []
        }

    context_parts = []
    sources = []

    for i, result in enumerate(search_results):
        context_parts.append(
            f"[Source {i+1}: {result['title']}, "
            f"Section: {result['section']}, "
            f"Page {result['page_number']}, "
            f"Chunk {result['chunk_index']}]\n"
            f"{result['text']}"
        )

        sources.append({
            "source_number": i + 1,
            "chunk_id": result["chunk_id"],
            "paper_id": result["paper_id"],
            "title": result["title"],
            "section": result["section"],
            "primary_category": result["primary_category"],
            "categories": result["categories"],
            "page_number": result["page_number"],
            "chunk_index": result["chunk_index"],
            "cosine_similarity": result["cosine_similarity"],
            "hybrid_score": result["hybrid_score"],
            "entity_overlap": result["entity_overlap"]
        })

    context = "\n\n".join(context_parts)
    answer = generate_answer(query, context)

    return {
        "query": query,
        "answer": answer,
        "sources": sources
    }

In [50]:
queries=[
    # # testing how it answers questions that are directly related to the paper
    # "What is the main purpose of the proposed HCI system?", 
    # # testing how it answer partially related questions
    # "What hardware and software requirements are needed to run the proposed HCI system?", 
    # # testing how it answers unrelated questions
    # "Who is the Last Airbender?",   
    # #entity based questions to test the entity normalization and filtering
    # "What role does the nose tip play in the interface?",
    # "How does blink detection work in the system?",
    # "What does the paper say about USB cameras?"
    "What is HCI and how is it used in the system?"
]

for q in queries:
    print("\n" + "="*50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_0: HCI

Entities for chunk 1002.2191v1_chunk_7: HCI

Entities for chunk 1002.2191v1_chunk_8: 1, 2010, HCI
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the described system, HCI is implemented by tracking the user's movements with a video camera and translating these movements into corresponding movements of the mouse pointer on the screen.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_0', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'page_number': 1, 'chunk_index': 0, 'cosine_similarity': 0.6645033955574036, 'hybrid_score': 0.6645033955574036, 'entity_overlap': 1}
{'source_number': 2, 'chunk_

### EXPERIMENTAL : RERANKING CHUNKS BASED ON ENTITY OVERLAP

In [55]:
entity_queries = [
    ("What is HCI and how is it used in the system?", {"use_hybrid": True}),
    # ("What role does the nose tip play in the interface?", {"use_hybrid": True}),
    # ("How does blink detection work in the system?", {"use_hybrid": True}),
    # ("What does the paper say about USB cameras?", {"use_hybrid": True}),
]

for q, opts in entity_queries:
    print("\n" + "=" * 50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3, **opts)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is HCI and how is it used in the system?'

Entities for chunk 1002.2191v1_chunk_0: HCI

Entities for chunk 1002.2191v1_chunk_7: HCI

Entities for chunk 1002.2191v1_chunk_8: 1, 2010, HCI
Answer:
 Human-Computer Interface (HCI) is the point of communication between a human and a computer, aiming to use human features to interact with the computer. In the system, HCI tracks the user's movements via a video camera and translates these movements into corresponding movements of the mouse pointer on the screen.

Sources:
{'source_number': 1, 'chunk_id': '1002.2191v1_chunk_0', 'paper_id': '1002.2191v1', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'section': 'I. INTRODUCTION', 'primary_category': 'cs.HC', 'categories': 'cs.HC, cs.CV, cs.MM', 'page_number': 1, 'chunk_index': 0, 'cosine_similarity': 0.6645033955574036, 'hybrid_score': 1.1645033955574036, 'entity_overlap': 1}
{'source_number': 2, 'chunk_id': '1002.2191v1_chunk_7', 'pap

This test is to explore whether NER-derived metadata could provide additional retrieval benefit (other than enriching metadata) through hybrid re-ranking. The use_hybrid=True to re rank chunks that have higher entity overlap after retrieving semantically as usual. However, there is currently no significant improvement in re-ranking the chunks based on entity overlap. The general semantic search flow using `retrieve_chunks` already performs good without this enhancement.

### EXPERIMENTAL : FILTER CATEGORY


This will be tested when we scale up documents once the categories vary among documents.